In [1]:
import pandas as pd
import numpy as np

# Define the file name and the data sheet
orig_data_file = r"climate_change_download_0.xls"
data_sheet = "Data"

# Read the data from the Excel file
data_orig = pd.read_excel(io=orig_data_file, sheet_name=data_sheet)

print("Shape of the original dataset:")
data_orig.shape

print("Available columns:")
data_orig.columns

print("Column data types:")
data_orig.dtypes

print("Overview of the first 5 rows:")
data_orig.head()

print("Descriptive statistics of the columns:")
data_orig.describe()

# Unique values inspection
data_orig['Series name'].unique()
data_orig['Series code'].unique()
data_orig['SCALE'].unique()
data_orig['Decimals'].unique()

# Filter out unwanted rows
data_clean = data_orig
print("Original number of rows:")
print(data_clean.shape[0])
data_clean = data_clean[data_clean['SCALE'] != 'Text']
print("Current number of rows:")
print(data_clean.shape[0])

# Drop unnecessary columns
print("Original number of columns:")
print(data_clean.shape[1])
data_clean = data_clean.drop(['Country name', 'Series code', 'SCALE', 'Decimals'], axis='columns')
print("Current number of columns:")
print(data_clean.shape[1])

# Replace missing indicators
data_clean.iloc[:, 2:] = data_clean.iloc[:, 2:].replace({'': np.nan, '..': np.nan})
data_clean2 = data_clean.applymap(lambda x: pd.to_numeric(x, errors='ignore'))
print("Column data types after transformation:")
data_clean2.dtypes

# Define shorter variable names
chosen_vars = {
    'Cereal yield (kg per hectare)': 'cereal_yield',
    'Foreign direct investment, net inflows (% of GDP)': 'fdi_perc_gdp',
    'Access to electricity (% of total population)': 'elec_access_perc',
    'Energy use per units of GDP (kg oil eq./$1,000 of 2005 PPP $)': 'en_per_gdp',
    'Energy use per capita (kilograms of oil equivalent)': 'en_per_cap',
    'CO2 emissions, total (KtCO2)': 'co2_ttl',
    'CO2 emissions per capita (metric tons)': 'co2_per_cap',
    'CO2 emissions per units of GDP (kg/$1,000 of 2005 PPP $)': 'co2_per_gdp',
    'Other GHG emissions, total (KtCO2e)': 'other_ghg_ttl',
    'Methane (CH4) emissions, total (KtCO2e)': 'ch4_ttl',
    'Nitrous oxide (N2O) emissions, total (KtCO2e)': 'n2o_ttl',
    'Droughts, floods, extreme temps (% pop. avg. 1990-2009)': 'nat_emerg',
    'Population in urban agglomerations >1million (%)': 'pop_urb_aggl_perc',
    'Nationally terrestrial protected areas (% of total land area)': 'prot_area_perc',
    'GDP ($)': 'gdp',
    'GNI per capita (Atlas $)': 'gni_per_cap',
    'Under-five mortality rate (per 1,000)': 'under_5_mort_rate',
    'Population growth (annual %)': 'pop_growth_perc',
    'Population': 'pop',
    'Urban population growth (annual %)': 'urb_pop_growth_perc',
    'Urban population': 'urb_pop'
}
data_clean2['Series name'] = data_clean2['Series name'].replace(to_replace=chosen_vars)
data_clean2.head()

# Reshape and merge all variables
chosen_cols = list(chosen_vars.values())
frame_list = []
for variable in chosen_cols:
    frame = data_clean2[data_clean2['Series name'] == variable]
    frame = frame.melt(id_vars=['Country code', 'Series name']).rename(
        columns={'Country code': 'country', 'variable': 'year', 'value': variable}
    ).drop(['Series name'], axis='columns')
    frame_list.append(frame)

from functools import reduce
all_vars = reduce(lambda left, right: pd.merge(left, right, on=['country', 'year'], how='outer'), frame_list)
all_vars.head()

# Check missing values
print("check the amount of missing values in each column")
all_vars.isnull().sum()

# Filter years
all_vars_clean = all_vars
years_count_missing = dict.fromkeys(all_vars_clean['year'].unique(), 0)
for ind, row in all_vars_clean.iterrows():
    years_count_missing[row['year']] += row.isnull().sum()
years_missing_sorted = dict(sorted(years_count_missing.items(), key=lambda item: item[1]))
print("missing values by year:")
for key, val in years_missing_sorted.items():
    print(key, ":", val)

print("Missing values before filtering years:")
print(all_vars_clean.isnull().sum().sum())
print("Rows before filtering years:")
print(all_vars_clean.shape[0])

all_vars_clean = all_vars_clean[(all_vars_clean['year'] >= 1991) & (all_vars_clean['year'] <= 2008)]
print("Missing values after filtering years:")
print(all_vars_clean.isnull().sum().sum())
print("Rows after filtering years:")
print(all_vars_clean.shape[0])

# Filter countries
countries_count_missing = dict.fromkeys(all_vars_clean['country'].unique(), 0)
for ind, row in all_vars_clean.iterrows():
    countries_count_missing[row['country']] += row.isnull().sum()
countries_missing_sorted = dict(sorted(countries_count_missing.items(), key=lambda item: item[1]))
print("missing values by country:")
for key, val in countries_missing_sorted.items():
    print(key, ":", val)

print("Missing values before filtering countries:")
print(all_vars_clean.isnull().sum().sum())
print("Rows before filtering countries:")
print(all_vars_clean.shape[0])

countries_filter = [key for key, val in countries_missing_sorted.items() if val < 90]
all_vars_clean = all_vars_clean[all_vars_clean['country'].isin(countries_filter)]
print("Missing values after filtering countries:")
print(all_vars_clean.isnull().sum().sum())
print("Rows after filtering countries:")
print(all_vars_clean.shape[0])

# Final cleaning
all_vars_clean.isnull().sum()

from itertools import compress
vars_bad = all_vars_clean.isnull().sum() > 20
all_vars_clean2 = all_vars_clean.drop(compress(all_vars_clean.columns, vars_bad), axis='columns')
print("Remaining missing values per column:")
print(all_vars_clean2.isnull().sum())

all_vars_clean3 = all_vars_clean2.dropna(axis='rows', how='any')
print("Remaining missing values per column:")
print(all_vars_clean3.isnull().sum())
print("Final shape of the cleaned dataset:")
print(all_vars_clean3.shape)

# Export to CSV
all_vars_clean3.to_csv('data_cleaned.csv', index=False)  

Shape of the original dataset:
Available columns:
Column data types:
Overview of the first 5 rows:
Descriptive statistics of the columns:
Original number of rows:
13512
Current number of rows:
10017
Original number of columns:
28
Current number of columns:
24


C:\Users\ASUS\AppData\Local\Temp\ipykernel_58156\2478220566.py:48: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data_clean.iloc[:, 2:] = data_clean.iloc[:, 2:].replace({'': np.nan, '..': np.nan})
C:\Users\ASUS\AppData\Local\Temp\ipykernel_58156\2478220566.py:49: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  data_clean2 = data_clean.applymap(lambda x: pd.to_numeric(x, errors='ignore'))
C:\Users\ASUS\AppData\Local\Temp\ipykernel_58156\2478220566.py:49: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  data_clean2 = data_clean.applymap(lambda x: pd.to_numeric(x, errors='ignore'))


Column data types after transformation:
check the amount of missing values in each column
missing values by year:
2005 : 1189
2000 : 1273
1995 : 1317
1990 : 1427
2007 : 1631
2006 : 1633
2004 : 1646
2008 : 1708
2003 : 1714
2002 : 1715
2001 : 1718
1999 : 1729
1998 : 1739
1997 : 1746
1996 : 1756
1994 : 1781
1993 : 1792
1992 : 1810
1991 : 1921
2009 : 2078
2010 : 3038
2011 : 4893
Missing values before filtering years:
41254
Rows before filtering years:
5126
Missing values after filtering years:
29818
Rows after filtering years:
4194
missing values by country:
AGO : 81
ARG : 81
AUS : 81
AUT : 81
BGD : 81
BGR : 81
BOL : 81
BRA : 81
CAN : 81
CHE : 81
CHL : 81
CHN : 81
CIV : 81
CMR : 81
COG : 81
COL : 81
CRI : 81
DEU : 81
DNK : 81
DOM : 81
ECU : 81
EGY : 81
EMU : 81
ESP : 81
FIN : 81
FRA : 81
GBR : 81
GHA : 81
GTM : 81
HND : 81
HUN : 81
IDN : 81
IND : 81
IRL : 81
ISR : 81
ITA : 81
JOR : 81
JPN : 81
KEN : 81
KOR : 81
LAC : 81
LMC : 81
LMY : 81
MAR : 81
MEX : 81
MIC : 81
MNA : 81
MOZ : 81
MYS : 8